In [1]:
import h5py
import numpy as np
import pandas as pd
import torch
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from lightning import Trainer
from lightning.pytorch.loggers import WandbLogger, TensorBoardLogger
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
)
from sklearn.metrics import matthews_corrcoef as mcc
from sklearn.metrics import f1_score as f1
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from torch.utils.data import DataLoader, Dataset
from torch.nn.functional import cross_entropy
from tqdm import tqdm

from src.data.lightning_glyco import ImpGroupedBatchSampler
from src.models import *


In [2]:
f1([0,1,1], [0,1,1])

1.0

In [3]:
def prepare_glyco_data(glyco_type: str):
    glyco_to_class = {
        'N': 1,
        'O': 2
        }
    glyco_class = glyco_type
    glyco_sites = pd.read_csv(f'data/glyco/{glyco_type}/{glyco_type}_train_RR.csv')
    val_glyco_sites = pd.read_csv(f'data/glyco/{glyco_type}/{glyco_type}_val_RR.csv')
    glyco_sites['split'] = 'train'
    val_glyco_sites['split'] = 'train'
    
    glyco_sites = pd.concat([glyco_sites, val_glyco_sites])
    #glyco_sites = glyco_sites[(glyco_sites['label'] == glyco_to_class[glyco_class]) | (glyco_sites['label'] == 0)]

    #glyco_sites["label"] = glyco_sites["label"].apply(lambda x: 1 if x >= 1 else 0)
    if glyco_class == 'N':
        glyco_sites = glyco_sites[glyco_sites["AA"] == 'N']
    else:
        glyco_sites = glyco_sites[(glyco_sites["AA"] == 'S') | (glyco_sites["AA"] == 'T')]
    #glyco_sites, labels = ros.fit_resample(glyco_sites, labels)
    glyco_sites.reset_index(drop=True, inplace=True)
    input_features = np.empty((len(glyco_sites), 2304))
    for idx, (pid, pos) in tqdm(enumerate(zip(glyco_sites['PID'], glyco_sites['Position']))):
        input_feature = np.empty(2304)
        with h5py.File(f'data/glyco/glyco_embeddings.h5', 'r') as p5, h5py.File(f'data/glyco/glyco_esm_embeddings.h5', 'r') as esm:
            try:
                #processed_pids = [pid.replace("-", "_").replace(".", "_") for pid in pids] 
                input_feature = np.concatenate([p5[pid.replace('-', '_').replace('.','_')][()][pos - 1], esm[pid.replace("_", "-")][()][pos - 1]])
            except:
                continue
            input_features[idx] = input_feature
    mask = np.all(input_features != 0, axis=1)
    input_features = input_features[mask]
    labels = np.array(glyco_sites['label'])[mask]
    print(np.sum(~mask))
    print(glyco_sites['label'][mask].value_counts())
    print(glyco_sites['label'].value_counts())
    input_features = input_features.astype(np.float32)
    labels = labels.astype(np.float16)
    return input_features, labels, glyco_sites[mask].reset_index(drop=True)

In [4]:
def prepare_data(sasa_or_bfactor: str):
    if sasa_or_bfactor == 'sasa':
        data_type = 'sasa'
    else:
        data_type = 'bfactor'
    train = pd.read_csv(f'data/e_prsa/{data_type}/train.csv')
    input_features = []
    ys = []
    pids = np.array(train['PID'].unique())
    for idx, pid in tqdm(enumerate(train['PID'].unique())):
        train_protein = train[train['PID'] == pid]
        y = train_protein['label'].values[0].astype(np.float32)
        input_feature = None
        with h5py.File(f'data/e_prsa/prott5_sasa_bfactor.h5', 'r') as p5, h5py.File(f'data/e_prsa/esm_sasa_bfactor.h5', 'r') as esm:
            try:
                #processed_pids = [pid.replace("-", "_").replace(".", "_") for pid in pids] 
                input_feature = np.concatenate([p5[pid.replace('-', '_').replace('.','_')][()], esm[pid.replace("_", "-")][()]])
            except:
                
                continue
            ys.append(y)
            input_features.append(input_feature)
    input_features = np.array(input_features, dtype=object)
    ys = np.array(ys, dtype=object)
    return input_features, ys, pids
    

In [5]:
class Glycodataset(Dataset):
    def __init__(self, X, y, pids):
        super().__init__()
        self.X = X
        self.y = y
        self.pids = pids
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.pids[idx]

class CVDataset(Dataset):
    def __init__(self, input_features, ys):
        self.input_features = input_features
        self.ys = ys

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        return self.input_features[idx], self.ys[idx]

In [13]:
def cv_train_glyco(glyco_class: str, 
                   input_features, 
                   labels, 
                   glyco_sites, 
                   us_ratio, 
                   os_ratio, 
                   batch_size,
                   hidden_size,
                   folds=5):
    train_idx = list(glyco_sites[glyco_sites['split'] == 'train'].index)
    pids = np.array(glyco_sites['PID'].values)
    
    train_X_o, train_y_o, train_pids_o = input_features[train_idx], labels[train_idx], pids[train_idx]
    
    
    cv_metrics = {}
    
    sfold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11442)
    for idx, (train_idx, val_idx) in enumerate(sfold.split(train_X_o, train_y_o, groups=train_pids_o)):
        train_X, train_y, train_pids = train_X_o[train_idx], train_y_o[train_idx], train_pids_o[train_idx]
        val_X, val_y, val_pids = train_X_o[val_idx], train_y_o[val_idx], train_pids_o[val_idx]
        rus = RandomUnderSampler(random_state=42)
        rus_r = RandomUnderSampler(sampling_strategy={0: int(train_y[train_y == 1].shape[0] * us_ratio), 
                                                    1: train_y[train_y == 1].shape[0]}, random_state=42)
        train_idx, train_y = rus_r.fit_resample(np.arange(len(train_y)).reshape((-1, 1)), train_y)
        train_idx = train_idx.squeeze()
        train_X = train_X[train_idx]
        train_pids = train_pids[train_idx]
        
        ros = RandomOverSampler(sampling_strategy={0: train_y[train_y == 0].shape[0], 
                                                    1: int(train_y[train_y == 1].shape[0] * os_ratio)}, random_state=42)
        train_idx, train_y = ros.fit_resample(np.arange(len(train_y)).reshape((-1, 1)), train_y)
        train_idx = train_idx.squeeze()
        train_X = train_X[train_idx]
        train_pids = train_pids[train_idx]
        
        val_idx, val_y = rus.fit_resample(np.arange(len(val_y)).reshape((-1, 1)), val_y)
        val_idx = val_idx.squeeze()
        val_X = val_X[val_idx]
        val_pids = val_pids[val_idx]
        
        train_dataset = Glycodataset(train_X, train_y, train_pids)
        val_dataset = Glycodataset(val_X, val_y, val_pids)
        train_dl = DataLoader(train_dataset, batch_sampler=ImpGroupedBatchSampler(train_pids, batch_size=batch_size))
        val_dl = DataLoader(val_dataset, 1, shuffle=False)
        glyco_model = GlycoModel(num_classes=2, 
                                 lr=0.0001,
                                 input_dim=2304, 
                                 num_hidden=hidden_size, 
                                 num_layers=2,
                                 class_weights=torch.tensor(train_y[train_y == 0].shape[0] / train_y[train_y == 1].shape[0]).to('cuda'))
        checkpoint_callback = ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1)
        tensor_b = TensorBoardLogger('tb_logs', 
                                     name=f'glyco_{glyco_class}_cv_{idx}', 
                                     default_hp_metric=False)
        #tensor_b = WandbLogger(name=f'glyco_{glyco_class}_cv_{idx}', project='protein_properties_cv')
        trainer = Trainer(max_epochs=30, 
                          enable_progress_bar=False, 
                          num_sanity_val_steps=1, 
                          logger=tensor_b,
                          callbacks=[EarlyStopping(monitor='val_loss', mode='min', patience=4), checkpoint_callback])        
        trainer.fit(glyco_model, train_dl, val_dl)
        
        y_pred_val = trainer.predict(ckpt_path='best', 
                                     dataloaders=DataLoader(val_dataset, 1, shuffle=False))
        val_loss = np.array([loss[1] for loss in y_pred_val]).mean()
        y_pred_val = np.array([pred[0] for pred in y_pred_val], dtype=np.float16)
        matt_val = mcc(val_y, y_pred_val)
        f1_val = f1(val_y, y_pred_val)
        acc_val = np.mean(val_y == y_pred_val)
        #loss_val = cross_entropy(torch.tensor(y_pred_val), torch.tensor(val_y).long())
        
        y_pred_train = trainer.predict(ckpt_path='best', 
                                       dataloaders=DataLoader(train_dataset, batch_size=1, shuffle=False))
        train_loss = np.array([loss[1] for loss in y_pred_train]).mean()
        y_pred_train = np.array([pred[0] for pred in y_pred_train], dtype=np.float16)
        matt_train = mcc(train_y, y_pred_train)
        f1_train = f1(train_y, y_pred_train)
        acc_train = np.mean(train_y == y_pred_train)
        #loss_train = cross_entropy(torch.tensor(y_pred_train), torch.tensor(train_y).long())
        
        cv_metrics[f'fold_{idx}'] = {
            'matt_train': matt_train,
            'f1_train': f1_train,
            'acc_train': acc_train,
            'loss_train': train_loss,
            'matt_val': matt_val,
            'f1_val': f1_val,
            'acc_val': acc_val,
            'loss_val': val_loss,
            'model': checkpoint_callback.best_model_path,
            'train_pred': y_pred_train,
            'val_pred': y_pred_val,
            'train_true': train_y,
            'val_true': val_y
        }
        
    return cv_metrics


In [7]:
input_features, labels, glyco_sites  = prepare_glyco_data('N')

0it [00:00, ?it/s]

15292it [02:01, 125.39it/s]


670
label
0    8815
1    5807
Name: count, dtype: int64
label
0    9228
1    6064
Name: count, dtype: int64


In [14]:
cv_metrics = cv_train_glyco('N', input_features, labels, glyco_sites, 0.8, 1.5, 128, [84])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: tb_logs/glyco_N_cv_0
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params
---------------------------------------
0 | model   | Sequential | 193 K 
1 | softmax | Softmax    | 0     
2 | sigmoid | Sigmoid    | 0     
---------------------------------------
193 K     Trainable params
0         Non-trainable params
193 K     Total params
0.775     Total estimated model params size (MB)
/home/d/PycharmProjects/protein_properties/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_z

In [15]:
cv_metrics

{'fold_0': {'matt_train': 0.6020570674415153,
  'f1_train': 0.8336597978531693,
  'acc_train': 0.8002258398419121,
  'loss_train': 0.4170309899386586,
  'matt_val': 0.5272873588000496,
  'f1_val': 0.751873071837814,
  'acc_val': 0.7626475548060708,
  'loss_val': 0.5209150394742043,
  'model': 'tb_logs/glyco_N_cv_0/version_0/checkpoints/epoch=28-step=2436.ckpt',
  'train_pred': array([0., 0., 0., ..., 1., 1., 1.], dtype=float16),
  'val_pred': array([0., 0., 1., ..., 1., 1., 1.], dtype=float16),
  'train_true': array([0., 0., 0., ..., 1., 1., 1.], dtype=float16),
  'val_true': array([0., 0., 0., ..., 1., 1., 1.], dtype=float16)},
 'fold_1': {'matt_train': 0.5889424917478169,
  'f1_train': 0.8238933841028082,
  'acc_train': 0.7907437081723065,
  'loss_train': 0.41809917388464396,
  'matt_val': 0.49868519921559473,
  'f1_val': 0.7266154541346589,
  'acc_val': 0.7466499162479062,
  'loss_val': 0.520663160774576,
  'model': 'tb_logs/glyco_N_cv_1/version_1/checkpoints/epoch=28-step=2407.ckpt

In [ ]:
# save results to disk
import pickle
with open('glyco_N_cv_results.pkl', 'wb') as f:
    pickle.dump(cv_metrics, f)

In [19]:
# load results from disk
import pickle
with open('glyco_N_cv_results.pkl', 'rb') as f:
    cv_metrics = pickle.load(f)

In [17]:
input_features, labels, glyco_sites  = prepare_glyco_data('O')

88282it [04:10, 351.84it/s]


8636
label
0    75727
1     3919
Name: count, dtype: int64
label
0    84087
1     4195
Name: count, dtype: int64


In [18]:
cv_metrics_o = cv_train_glyco('O', input_features, labels, glyco_sites, 1.0, 1.7, 16, [46])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: tb_logs/glyco_O_cv_0
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name    | Type       | Params
---------------------------------------
0 | model   | Sequential | 106 K 
1 | softmax | Softmax    | 0     
2 | sigmoid | Sigmoid    | 0     
---------------------------------------
106 K     Trainable params
0         Non-trainable params
106 K     Total params
0.424     Total estimated model params size (MB)
/home/d/PycharmProjects/protein_properties/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:432: PossibleUserWarning: The dataloader, val_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 8 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_z

In [19]:
import pickle
with open('glyco_O_cv_results.pkl', 'wb') as f:
    pickle.dump(cv_metrics_o, f)

In [20]:
cv_metrics_o

{'fold_0': {'matt_train': 0.7514302928301789,
  'f1_train': 0.8984081041968162,
  'acc_train': 0.8778563971697019,
  'loss_train': 0.41595969439631425,
  'matt_val': 0.5624466408799765,
  'f1_val': 0.752851711026616,
  'acc_val': 0.7761707988980716,
  'loss_val': 0.5144573453372503,
  'model': 'tb_logs/glyco_O_cv_0/version_0/checkpoints/epoch=22-step=12397.ckpt',
  'train_pred': array([0., 0., 0., ..., 1., 1., 1.], dtype=float16),
  'val_pred': array([0., 0., 1., ..., 1., 1., 1.], dtype=float16),
  'train_true': array([0., 0., 0., ..., 1., 1., 1.], dtype=float16),
  'val_true': array([0., 0., 0., ..., 1., 1., 1.], dtype=float16)},
 'fold_1': {'matt_train': 0.6029013295992568,
  'f1_train': 0.8111669458403127,
  'acc_train': 0.7885457046392397,
  'loss_train': 0.43776792273665127,
  'matt_val': 0.5785448030627568,
  'f1_val': 0.7720670391061453,
  'acc_val': 0.786833855799373,
  'loss_val': 0.5185740758881923,
  'model': 'tb_logs/glyco_O_cv_1/version_0/checkpoints/epoch=6-step=3500.ckpt